# Setup

In [1]:
import time
from transformers import GenerationConfig # TODO: use generation config
from datasets import load_dataset
import torch
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
import os

import pandas as pd

from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments
)
from typing import List, Union

from trl import SFTTrainer

/home/ubuntu/anaconda3/lib/python3.10/site-packages/trl/trainer/ppo_config.py:141: UserWarning: The `optimize_cuda_cache` arguement will be deprecated soon, please use `optimize_device_cache` instead.
  warnings.warn(


# Загружаем датасет

In [2]:
train_df_path = '/home/stas/vaML/titanic/train.csv'
test_X_path = '/home/stas/vaML/titanic/test.csv'
test_y_path = '/home/stas/vaML/titanic/gender_submission.csv'

train_df = pd.read_csv(train_df_path)
test_X = pd.read_csv(test_X_path)
test_y = pd.read_csv(test_y_path)

train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train_df.columns.values

array(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'], dtype=object)

In [4]:
text_columns = list(train_df.select_dtypes(include=['object']).columns)
numeric_columns = list(train_df.select_dtypes(include=['number']).columns)

train_df_columns = list(train_df.columns)
test_X_columns = list(test_X.columns)
test_y_columns = list(test_y.columns)

categorical_columns = ['Survived', 'Pclass', 'Sex', 'Embarked']
dataset_name = "titanic passengers survival"
dataset_description = "passengers survived the Titanic shipwreck"
dataset_goal = "predict which passengers survived the Titanic shipwreck"
target_column = "Survived"

In [5]:
print(text_columns)
print(numeric_columns)

['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


# Prompt variant 1

In [8]:
categorical_lines = [f"{col_name}: {list(train_df[col_name].unique())}" for col_name in categorical_columns]

prompt_lines = [
    f"Assume we have a dataset called '{dataset_name}', which describes {dataset_description}, and the task is to {dataset_goal}.",
    f"The dataset contains the following text columns: {text_columns}",
    f"The dataset contains the following numeric columns: {numeric_columns}",
    f"Listed below are the categorical columns with all their unique values"
] + categorical_lines + [
    f"The target column is {target_column}"
    f"Using the XGBoost library in python, show how you would solve this task."
]

prompt = "\n".join(prompt_lines)
print(prompt)

Assume we have a dataset called 'titanic passengers survival', which describes passengers survived the Titanic shipwreck, and the task is to predict which passengers survived the Titanic shipwreck.
The dataset contains the following text columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
The dataset contains the following numeric columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Listed below are the categorical columns with all their unique values
Survived: [0, 1]
Pclass: [3, 1, 2]
Sex: ['male', 'female']
Embarked: ['S', 'C', 'Q', nan]
The target column is SurvivedUsing the XGBoost library in python, show how you would solve this task.


# Prompt variant 2

In [6]:
categorical_lines = [f"{col_name}: {list(train_df[col_name].unique())}" for col_name in categorical_columns]

prompt_lines = [
    f"Assume we have a dataset called '{dataset_name}', which describes {dataset_description}, and the task is to {dataset_goal}.",
    f"",
    f"The train dataset is stored at: {train_df_path}",
    f"It contains: {train_df_columns} columns.",
    f"",
    f"The test dataset without target is stored at: {test_X_path}",
    f"It contains: {test_X_columns} columns.",
    f"",
    f"The test dataset target is stored at: {test_y_path}",
    f"It contains: {test_y_columns} columns.",
    f"",
    f"The dataset contains the following string columns: {text_columns}",
    f"The dataset contains the following numeric columns: {numeric_columns}",
    f"Listed below are the categorical columns with all their unique values",
    ] + categorical_lines + [
    f"The target column is '{target_column}'.",
    f"",
    f"Using the XGBoost library in python, show how you would solve this task. Import the necessary libraries.",
    f"Make sure to handle missing values, convert categorical columns into numeric format.",
    f"Use one-hot encoding for non-binary categorical columns, label encoding for binary columns.",
    f"Do not split datasets as they are already split.",
    f"Define the optimal XGBoost parameters, train the model on the training data.",
    f"Make predictions on the test, calculate performance metrics like accuracy, precision, recall, etc.",
    f"Only return the code without any extra comments or explainations.",
]

prompt = "\n".join(prompt_lines)
print(prompt)

Assume we have a dataset called 'titanic passengers survival', which describes passengers survived the Titanic shipwreck, and the task is to predict which passengers survived the Titanic shipwreck.

The train dataset is stored at: /home/stas/vaML/titanic/train.csv
It contains: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'] columns.

The test dataset without target is stored at: /home/stas/vaML/titanic/test.csv
It contains: ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'] columns.

The test dataset target is stored at: /home/stas/vaML/titanic/gender_submission.csv
It contains: ['PassengerId', 'Survived'] columns.

The dataset contains the following string columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
The dataset contains the following numeric columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Listed below are the categorical

# Загружаем модель и проверяем ее работу

In [8]:
base_model_id = "NousResearch/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, #?
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
    low_cpu_mem_usage=True #?
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(base_model_id, 
                                             quantization_config = bnb_config, 
                                             device_map="auto")

# model.config.use_cache = False

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
def inference(text: Union[str, List[str], List[int]], 
              generation_config: dict,
              model, tokenizer, 
              max_input_tokens=1000, max_output_tokens=100):
  
    # токенизируем вход
  input_ids = tokenizer.encode(
          text,
          return_tensors="pt",
          truncation=True,
          max_length=max_input_tokens
  ).to('cuda')

  # генерируем
  generated_result = model.generate(
    input_ids=input_ids,
    max_length=max_output_tokens,
    **generation_config
  )

  # Декодируем
  generated_result = tokenizer.batch_decode(generated_result, 
                                                      skip_special_tokens=True)

  # Обрезаем промпт
  generated_text_answer = generated_result[0][len(text):]

  return generated_text_answer

In [11]:
# генерируем из модели 
generation_config = {
    "do_sample": True,
    "temperature": 0.8,
    "top_p": 0.8,
    "top_k": 20,
}

answer = inference(prompt,
          generation_config,
          model, tokenizer,
          max_output_tokens=1000)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [12]:
print(answer)

```python
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
train_data = pd.read_csv('/home/stas/vaML/titanic/train.csv')
test_data = pd.read_csv('/home/stas/vaML/titanic/test.csv')
submission_data = pd.read_csv('/home/stas/vaML/titanic/gender_submission.csv')

# Define the preprocessing steps
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),